# 📈 Robot Axis Regression

In this notebook, we attempt to predict the current of **Axis #6** based on **Axis #1**.
High correlation between axes suggests coordinated movement, and deviations might imply mechanical issues.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import os

# Load Processed Data
df = pd.read_csv('../data/processed/robot_current_clean.csv')
print(f"Loaded Data: {df.shape}")

**📝 Explanation:**
We load the pre-processed robot data. We use `train_test_split` to create a training set for the model to learn from and a test set to validate its predictions on unseen robot cycles.

In [ ]:
# Feature Selection
# Predicting Axis 6 (Wrist/Hand often) from Axis 1 (Base)
X = df[['Axis #1']].values
y = df['Axis #6'].values

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

**📝 Explanation:**
We select our variables. We treat `Axis #1` (Base) as the independent variable X, and `Axis #6` (Wrist) as the dependent variable y, assuming a mechanical coupling or coordinated task relationship.

## 2. Implementation from Scratch (Gradient Descent)
We implement Linear Regression ($y = wx + b$) using Gradient Descent.

In [ ]:
class LinearRegressionScratch:
    def __init__(self, learning_rate=0.0001, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.cost_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iterations):
            y_predicted = np.dot(X, self.weights) + self.bias
            
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            cost = self.compute_cost(X, y)
            self.cost_history.append(cost)

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

    def compute_cost(self, X, y):
        y_pred = np.dot(X, self.weights) + self.bias
        return (1 / (2 * len(y))) * np.sum((y_pred - y) ** 2)

# Train Model
model_scratch = LinearRegressionScratch(learning_rate=0.001, n_iterations=2000)
model_scratch.fit(X_train, y_train)

print(f"Scratch Model - Weight: {model_scratch.weights[0]:.4f}, Bias: {model_scratch.bias:.4f}")

**📝 Explanation:**
We build the Linear Regression logic from scratch. The `fit` method uses Gradient Descent to find the optimal line $y = wx + b$ that best describes the relationship between the two axes' currents.

In [ ]:
# Plot Cost History
plt.plot(model_scratch.cost_history)
plt.title('Cost History (Training Error)')
plt.xlabel('Iterations')
plt.ylabel('Cost')
plt.show()

**📝 Explanation:**
We inspect the learning process. The Cost History graph shows us how the error decreased over 2000 iterations, confirming the model converged to a solution.

## 3. Scikit-Learn Implementation

In [ ]:
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)
print(f"Sklearn Model - Weight: {model_sklearn.coef_[0]:.4f}, Bias: {model_sklearn.intercept_:.4f}")

**📝 Explanation:**
We train a reference `Scikit-Learn` model. This allows us to verify that our `scratch` calculation is performing correctly and yielding accurate weights.

## 4. Visualization & Comparison

In [ ]:
y_pred_scratch = model_scratch.predict(X_test)
y_pred_sklearn = model_sklearn.predict(X_test)

rmse_scratch = np.sqrt(mean_squared_error(y_test, y_pred_scratch))
rmse_sklearn = np.sqrt(mean_squared_error(y_test, y_pred_sklearn))

plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', alpha=0.3, label='Actual Data')
plt.plot(X_test, y_pred_scratch, color='red', label=f'Scratch (RMSE={rmse_scratch:.4f})')
plt.plot(X_test, y_pred_sklearn, color='green', linestyle='--', label=f'Sklearn (RMSE={rmse_sklearn:.4f})')
plt.title('Linear Regression: Axis #1 vs Axis #6 Current')
plt.xlabel('Axis #1 Current')
plt.ylabel('Axis #6 Current')
plt.legend()
plt.show()

**📝 Explanation:**
We visualize the predictions. The scatter plot shows the actual current readings (Blue) vs our model predictions (Red/Green lines). The tight alignment confirms a linear relationship exists.

In [ ]:
# Save Results
os.makedirs('../experiments', exist_ok=True)

results = pd.DataFrame({
    'Model': ['Linear Regression (Scratch)', 'Linear Regression (Sklearn)'],
    'RMSE': [rmse_scratch, rmse_sklearn]
})

results.to_csv('../experiments/results.csv', index=False)
print("Results saved to experiments/results.csv")

**📝 Explanation:**
We save the Root Mean Squared Error (RMSE) to a persistent CSV file in the `experiments` folder for reporting and tracking model performance.